John S. Anvik, 2026  
University of Lethbridge Student ID: 001224816

In [1]:
# Import packages
# install.packages("slider")
library(tidyverse)
library(slider)
options(repr.matrix.max.cols = Inf) # display all columns in tibbles

Warning message:
"package 'tidyverse' was built under R version 4.5.3"
Warning message:
"package 'ggplot2' was built under R version 4.5.3"
Warning message:
"package 'tibble' was built under R version 4.5.3"
Warning message:
"package 'tidyr' was built under R version 4.5.3"
Warning message:
"package 'readr' was built under R version 4.5.3"
Warning message:
"package 'purrr' was built under R version 4.5.3"
Warning message:
"package 'dplyr' was built under R version 4.5.3"
Warning message:
"package 'lubridate' was built under R version 4.5.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<

In [2]:
# A lookup table to easily access IDs /ID numbers for each neighbourhood
focal_males <- tibble::tribble(
    ~neighbourhood, ~male_number, ~id,
    "north",  "1", "LbOW",
    "north",  "2", "RbDgY",
    "north",  "3", "YKY",
    "north",  "4", "TbTbLg",
    "north",  "5", "DgTbO",
    "north",  "6", "RRO",
    "centre", "1", "TbWY",
    "centre", "2", "KRDg",
    "centre", "3", "OPR",
    "centre", "4", "WLpTb",
    "centre", "5", "KLgK",
    "centre", "6", "DgTbO"
)

In [3]:
# minizord.csv already removed non-songs, post-dawn, unmeasureable songs, acoustic outliers
# see "Data Preprocessing/01 preprocessing.ipynb"

# Import data
raw_data <- read_csv("data/minizord.csv")
raw_data <- raw_data %>% filter(ID != "OPR")
# EXCLUDE SOUTH NEIGHBOURHOOD
raw_data <- raw_data %>% filter(neighbourhood != "south")

neighbourhoods <- sort(unique(na.omit(raw_data$neighbourhood)))
focal_ids <- unique(na.omit(raw_data$ID))

Rows: 11539 Columns: 53
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (4): ID, IST, neighbourhood, Location_Number
dbl  (46): time_rel_sunrise, UST, D_M1_M2, D_M1_M3, D_M1_M4, D_M1_M5, D_M1_M...
lgl   (1): RepeatST
dttm  (1): DateTime
date  (1): date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [4]:
# Which song types are present in each focal male's neighbourhood(s)?
for (n in raw_data %>% pull(neighbourhood) %>% unique()) {
    all_song_types <- raw_data %>% 
        filter(neighbourhood == n) %>%
        pull(UST) %>%
        unique() %>%
        sort()
    cat("All songs types in ", n, "' neighbourhood", "(", length(all_song_types), " songs)\n")
    print(all_song_types)
}

All songs types in  north ' neighbourhood ( 65  songs)
 [1]   1   3   4   5   6   7   8   9  10  11  12  13  15  16  17  18  19  20  21
[20]  22  25  26  27  28  29  30  31  32  33  35  36  60  61  62  63  64  90  91
[39]  92  93  94  95  96  97  98 100 115 116 117 118 119 120 153 154 155 156 157
[58] 165 166 167 168 169 170 171 172
All songs types in  centre ' neighbourhood ( 59  songs)
 [1]   1   3   4   5   6   7   8   9  10  11  12  13  14  15  20  23  24  25  26
[20]  28  30  32  33  55  56  57  58  59  83  84  85  86  87  88  89 101 102 103
[39] 104 105 106 107 108 109 110 111 112 113 114 153 154 155 158 159 160 161 162
[58] 163 164


In [5]:
data_list <- list()

for (focal_id in focal_ids) {
    cat(focal_id, focal_males %>% filter(id == focal_id) %>% pull(neighbourhood) %>% unique(), "\n")
    data_to_append <- raw_data %>% filter(neighbourhood %in% (focal_males %>% filter(id == focal_id) %>% pull(neighbourhood)))
    data_list <- append(data_list, list(data_to_append))
}
names(data_list) <- focal_ids

print(length(data_list))

DgTbO north centre 
KLgK centre 
KRDg centre 
LbOW north 
RRO north 
RbDgY north 
TbTbLg north 
TbWY centre 
WLpTb centre 
YKY north 
[1] 10


In [6]:
# Which song types are present in each focal male's neighbourhood(s)?
# Song types present in each focal male's neighbourhood(s); used by the n_ust_* loops below.
# (DgTbO is in both neighbourhoods, so his list is the union of the two.)
song_types_by_id <- list()

for (id in focal_ids) {
    cat("All songs types in ", id, "'s neighbourhood", "\n")
    all_song_types <- sort(unique(na.omit(data_list[[id]]$UST)))
    song_types_by_id[[id]] <- all_song_types
    print(all_song_types)
    print
}

All songs types in  DgTbO 's neighbourhood 
  [1]   1   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18  19
 [19]  20  21  22  23  24  25  26  27  28  29  30  31  32  33  35  36  55  56
 [37]  57  58  59  60  61  62  63  64  83  84  85  86  87  88  89  90  91  92
 [55]  93  94  95  96  97  98 100 101 102 103 104 105 106 107 108 109 110 111
 [73] 112 113 114 115 116 117 118 119 120 153 154 155 156 157 158 159 160 161
 [91] 162 163 164 165 166 167 168 169 170 171 172
All songs types in  KLgK 's neighbourhood 
 [1]   1   3   4   5   6   7   8   9  10  11  12  13  14  15  20  23  24  25  26
[20]  28  30  32  33  55  56  57  58  59  83  84  85  86  87  88  89 101 102 103
[39] 104 105 106 107 108 109 110 111 112 113 114 153 154 155 158 159 160 161 162
[58] 163 164
All songs types in  KRDg 's neighbourhood 
 [1]   1   3   4   5   6   7   8   9  10  11  12  13  14  15  20  23  24  25  26
[20]  28  30  32  33  55  56  57  58  59  83  84  85  86  87  88  89 101 102 103
[39] 104 105

In [7]:
# Repertoire of each focal male??
for (id in focal_ids) {
    repertoire <- data_list[[id]] %>% filter(ID==id)
    repertoire <- repertoire %>% pull(UST) %>% unique() %>% sort()
    cat("All songs types in ", id, "'s repertoire: (", length(repertoire), " songs)\n")
    print(repertoire)
    print
}

All songs types in  DgTbO 's repertoire: ( 28  songs)
 [1]   1   4   5   6   7   8  10  11  12  13  15  20  25  30  32  33 153 154 155
[20] 156 157 158 159 160 161 162 163 164
All songs types in  KLgK 's repertoire: ( 19  songs)
 [1]  1  3  4  5  6  7  8  9 10 11 12 13 14 15 55 56 57 58 59
All songs types in  KRDg 's repertoire: ( 26  songs)
 [1]   1   4   6   8   9  10  12  13  15  20  23  24  25  28  33 104 105 106 107
[20] 108 109 110 111 112 113 114
All songs types in  LbOW 's repertoire: ( 18  songs)
 [1]   3   5   8   9  10  12  13  16  17  20  21  29 115 116 117 118 119 120
All songs types in  RRO 's repertoire: ( 19  songs)
 [1]   3   5   6   8   9  10  12  13  15  17  19  20  28  29  30  31  97  98 100
All songs types in  RbDgY 's repertoire: ( 21  songs)
 [1]  1  3  5  8  9 13 16 17 18 19 20 21 22 27 90 91 92 93 94 95 96
All songs types in  TbTbLg 's repertoire: ( 19  songs)
 [1]  1  3  5  8  9 10 13 16 17 18 19 20 21 22 60 61 62 63 64
All songs types in  TbWY 's repertoire: 

In [8]:
# # Which song types are present in each focal male's neighbourhood(s)?
# data_list %>% map(~ print(sort(na.omit(unique(.x$UST)))))

In [9]:
for (focal_id in focal_ids) {
    repertoire <- data_list[[focal_id]] %>% filter(ID == focal_id) %>% arrange(UST) %>% pull(UST) %>% unique
    data_list[[focal_id]] <- data_list[[focal_id]] %>% mutate(UST = if_else(ID == focal_id | UST %in% repertoire, UST, -1))  # preserve focal songs; recode only other males' out-of-repertoire songs
}

In [10]:
data_list <- data_list %>% 
    map(~ .x %>% mutate(UST = replace_na(UST, 0))) %>% # NA song type values mess up our feature engineering below, we'll set them to zero
    map(~ .x %>%     mutate(across(contains("UST"), ~ as.factor(.x)), 
            does_it_match = as.factor(does_it_match),
            number_of_matches = as.integer(number_of_matches)))

In [11]:
for (focal_id in focal_ids) {
    print(focal_id)
    data_list[[focal_id]] <- data_list[[focal_id]] %>%
        arrange(DateTime) %>%
        # Grouping variable for the "since it last sang" counts: how many focal songs came BEFORE this row
        # (cumulative over all days; it increments on the row AFTER the focal male's song)
        mutate(focal_song_group = replace_na(lag(cumsum(ID == focal_id), n = 1), 0)) %>%
        # The focal male's nth song of the day (1 = first song that day; on other males' rows: focal songs so far today)
        group_by(date) %>%
        mutate(focal_male_song_number = cumsum(ID == focal_id)) %>%
        ungroup()
}

[1] "DgTbO"
[1] "KLgK"
[1] "KRDg"
[1] "LbOW"
[1] "RRO"
[1] "RbDgY"
[1] "TbTbLg"
[1] "TbWY"
[1] "WLpTb"
[1] "YKY"


In [12]:
# Create a column for the number of times the focal male has heard each song type since it last sang

for (focal_id in focal_ids) {
    print(focal_id)
    data_list[[focal_id]] <- data_list[[focal_id]] %>% arrange(DateTime) %>% group_by(focal_song_group)
    for (song_type in song_types_by_id[[focal_id]]) {
        n_ust <- sprintf("n_ust_%i", song_type)
        data_list[[focal_id]] <- data_list[[focal_id]] %>% mutate("{n_ust}" := 0) %>%
        mutate("{n_ust}" := cumsum((UST == song_type) & (ID != focal_id))) # Count each song type, don't count M1's current song
    }
    # Create a column for the total number of other males' songs that M1 has heard since it last sang
    data_list[[focal_id]] <- data_list[[focal_id]] %>% mutate(n_ust_total = cumsum(ID != focal_id))
    data_list[[focal_id]] <- data_list[[focal_id]] %>% ungroup()
    # write_csv(data_list[[focal_id]], "data/temp.csv")
}

[1] "DgTbO"
[1] "KLgK"
[1] "KRDg"
[1] "LbOW"
[1] "RRO"
[1] "RbDgY"
[1] "TbTbLg"
[1] "TbWY"
[1] "WLpTb"
[1] "YKY"


In [13]:
# Number of times the focal male has heard each song type in last <WINDOW_SIZE> seconds

WINDOW_SIZE <- 60
for (focal_id in focal_ids) {
  
  # Start with the focal male's data
  main_df <- data_list[[focal_id]] %>% arrange(DateTime)
  

  for (song_type in song_types_by_id[[focal_id]]) {
        window_col <- sprintf("n_ust_%i_window", song_type)

    main_df <- main_df %>%
      mutate(
        is_others_song = (ID != focal_id & UST == song_type)
      ) %>%
      mutate(
        !!window_col := slide_index_sum(x = is_others_song, i = DateTime, before = WINDOW_SIZE)
      ) %>%
      select(-is_others_song) # Remove temporary columns
  }
  
  data_list[[focal_id]] <- main_df
}

In [14]:
# Calculate time since each Male's last song
# Male i's nth last song

N_MAX <- 3 # Find 1st - 3rd last song types

for (focal_id in focal_ids) {
        focal_neighbourhood <- data_list[[focal_id]] %>% pull(neighbourhood) %>% unique
        # print(focal_neighbourhood)
        for (n in 1:N_MAX) {
                for (i in 1:6) {
                        id <- focal_males %>% filter(neighbourhood == focal_neighbourhood) %>% filter(male_number == i) %>% pull(id)
                        n_last_song <- sprintf("M%i_%i_last_UST", i, n)
                        time_last_song <- sprintf("time_M%i_%i_last_song", i, n)
                        time_since_last_song <- sprintf("time_since_M%i_%i_last_song", i, n)
                        data_list[[focal_id]] <- data_list[[focal_id]] %>%
                        # Sort by ID and DateTime
                                group_by(ID) %>%
                                arrange(DateTime) %>%
                        # Take the Nth last UST, set it to MX_Nth_last_UST
                                mutate("{n_last_song}" := if_else(ID == id, lag(UST, n-1), NA)) %>%
                                # mutate("{time_last_song}" := if_else(ID == id, time_rel_sunrise, NA)) %>%
                                mutate("{time_last_song}" := if_else(ID == id, lag(time_rel_sunrise, n-1), NA)) %>%
                                ungroup() %>% 
                        # Sort by DateTime, forward fill
                                group_by(date) %>%
                                arrange(DateTime) %>%
                                fill({n_last_song}, .direction = "down") %>%
                                fill({time_last_song}, .direction = "down") %>%
                        # Since we don't want the current song to count as the 1st last song, recalculate last songs for rows where ID matches MX
                                group_by(ID, date) %>%
                                arrange(DateTime) %>%
                                mutate("{n_last_song}" := if_else(ID == id, lag(UST, n), !!sym({n_last_song}))) %>%
                                mutate("{time_last_song}" := if_else(ID == id, lag(time_rel_sunrise, n), !!sym({time_last_song}))) %>%
                                ungroup() %>% 
                        # Calculate time since Nth last song
                                group_by(date) %>%
                                mutate("{time_since_last_song}" := time_rel_sunrise - !!sym({time_last_song})) %>%
                                ungroup()
                }
        }
}

In [15]:
WINDOW_SIZE <- 60

for (focal_id in focal_ids) {
    if (focal_id == "OPR") {next}
    focal_neighbourhood <- data_list[[focal_id]] %>% pull(neighbourhood) %>% unique()
    data_list[[focal_id]] <- data_list[[focal_id]] %>% arrange(DateTime)
    
    # R1 Create a feature counting how many songs were sung by focal male in time window
    R1 <- sprintf("focal_last_%i_seconds_n_songs", WINDOW_SIZE)
    data_list[[focal_id]] <- data_list[[focal_id]] %>%
        mutate("{R1}" := slide_index_int(
            .x = ID, .i = DateTime, .before = seconds(WINDOW_SIZE),
            .f = ~sum(.x == focal_id)
        ))

    # R2 Create a feature for each neighbour counting how many songs they sung in time window
    # C Create a feature for each neighbour counting all songs that were sung in the last 1-minute of shared type with neighbour
    
    for (i in 1:6) {
        neighbor_id <- focal_males %>% filter(neighbourhood == focal_neighbourhood, male_number == i) %>% pull(id)
        if ((neighbor_id == focal_id) ||( neighbor_id == "OPR")) {next}
        
        R2 <- sprintf("M%i_last_%i_seconds_n_songs", i, WINDOW_SIZE)
        C  <- sprintf("M%i_last_%i_seconds_shared_songs", i, WINDOW_SIZE)
        
        result <- data_list[[focal_id]] %>%
            mutate(
                temp_list = slide_index(
                    .x = pick(everything()), # 'pick' replaces cur_data_all safely
                    .i = DateTime,
                    .before = seconds(WINDOW_SIZE),
                    .after = 0,
                    .f = function(window_df) {
                        focal_songs <- window_df$UST[window_df$ID == focal_id]
                        neighbour_songs <- window_df$UST[window_df$ID == neighbor_id]
                        
                        list(
                            n_count = length(neighbour_songs),
                            shared_count = sum(focal_songs %in% intersect(focal_songs, neighbour_songs))
                        )
                    }
                )
            )
        
        # Extract the list into proper columns
        data_list[[focal_id]][[R2]] <- map_int(result$temp_list, "n_count")
        data_list[[focal_id]][[C]]  <- map_int(result$temp_list, "shared_count")
    }
}

In [16]:
# Use the three columns we just created to calculate "attention" value
for (focal_id in focal_ids) {
    if (focal_id == "OPR") {next}
    focal_neighbourhood <- data_list[[focal_id]] %>% pull(neighbourhood) %>% unique()
    R1 <- sprintf("focal_last_%i_seconds_n_songs", WINDOW_SIZE)
    for (i in 1:6) {
        neighbor_id <- focal_males %>% filter(neighbourhood == focal_neighbourhood, male_number == i) %>% pull(id)
        if ((neighbor_id == focal_id) ||( neighbor_id == "OPR")) {next}

        attention_col <- sprintf("Attention_on_M%i", i)
        R2 <- sprintf("M%i_last_%i_seconds_n_songs", i, WINDOW_SIZE)
        C  <- sprintf("M%i_last_%i_seconds_shared_songs", i, WINDOW_SIZE)

        data_list[[focal_id]] <- data_list[[focal_id]] %>% mutate(
            "{attention_col}" := (2*!!sym({C})) / (!!sym(R1) + !!sym(R2)),
        )
    }
}

In [17]:
# Determine "Attention Target" by comparing shared songs sung in last <WINDOW_SIZE>

id_lookup <- focal_males %>%
  mutate(male_number = as.character(male_number)) %>%
  select(id, male_number, neighbourhood)

for (focal_id in focal_ids) {
  if (focal_id == "OPR") {next}
  attention_data <- data_list[[focal_id]] %>% select(contains("Attention_on_M"))
  
  data_list[[focal_id]] <- data_list[[focal_id]] %>%
    mutate(
      highest_attention = do.call(pmax, c(attention_data, na.rm = TRUE)),

      attention_target_male_number = if_else(
        highest_attention > 0,
        sub("Attention_on_M", "", names(attention_data)[max.col(attention_data, ties.method = "first")]),
        NA_character_
      )
    ) %>%

    left_join(
      id_lookup, 
      by = c("attention_target_male_number" = "male_number", "neighbourhood" = "neighbourhood"),
      suffix = c("", "_neighbor")
    ) %>%
    rename(attention_target_id = id) %>%
    # If no attention target, use string "None" instead of NA values (needed so row isn't dropped by models)
    mutate(attention_target_id = replace_na(attention_target_id, "None"))
}

In [18]:
# Count the number of each song type sang by the attention target and by all other males in the time window

WINDOW_SIZE <- 60
for (focal_id in focal_ids) {
  if (focal_id == "OPR") {next}
  # Start with the focal male's data
  main_df <- data_list[[focal_id]] %>% arrange(DateTime)
  
  # Create a "Lookup" of every song ever sung

  all_songs <- main_df %>% 
    select(DateTime, SingerID = ID, SongType = UST)

  for (song_type in song_types_by_id[[focal_id]]) {
    
    target_col <- sprintf("n_ust_%i_target_window", song_type)
    others_col <- sprintf("n_ust_%i_others_window", song_type)

    main_df <- main_df %>%
      mutate(
        # Temporary logical columns for this specific song type
        is_target_song = (ID == attention_target_id & UST == song_type),
        is_others_song = (ID != focal_id & ID != attention_target_id & UST == song_type)
      ) %>%
      mutate(
        # slide_index_sum is a "vectorized" version of the window sum
        !!target_col := slide_index_sum(x = is_target_song, i = DateTime, before = WINDOW_SIZE),
        !!others_col := slide_index_sum(x = is_others_song, i = DateTime, before = WINDOW_SIZE)
      ) %>%
      select(-is_target_song, -is_others_song) # Remove temporary columns
  }
  
  data_list[[focal_id]] <- main_df
}

In [19]:
for (song_type in all_song_types) {
    
    target_col <- sprintf("n_ust_%i_target_window", song_type)
    others_col <- sprintf("n_ust_%i_others_window", song_type)

    main_df <- main_df %>%
      mutate(
        # Temporary logical columns for this specific song type
        is_target_song = (ID == attention_target_id & UST == song_type),
        is_others_song = (ID != focal_id & ID != attention_target_id & UST == song_type)
      ) %>%
      mutate(
        # slide_index_sum is a "vectorized" version of the window sum
        !!target_col := slide_index_sum(x = is_target_song, i = DateTime, before = WINDOW_SIZE),
        !!others_col := slide_index_sum(x = is_others_song, i = DateTime, before = WINDOW_SIZE)
      ) %>%
      select(-is_target_song, -is_others_song) # Remove temporary columns
  }

In [20]:
get_mode <- function(x) {
  if (length(x) == 0) return(NA)
  tab <- table(x)
  # Returns the name of the attribute with the maximum count
  names(tab)[which.max(tab)]
}

In [21]:
# The n_max most common song types sung by neighbour z within one window, padded with NA.
# x = song types in the window, y = singer IDs in the window (same length), z = the neighbour we want to count.
# Song types are counted as observed values only: table() on a factor would also list unused levels with a count of 0,
# which used to be returned as if they were real song types. Ties are broken by song type (ascending).
get_top_modes <- function(x, y, z, n_max = 3) {
    songs  <- suppressWarnings(as.numeric(as.character(x[which(y == z)])))
    songs  <- songs[!is.na(songs)]
    counts <- sort(table(songs), decreasing = TRUE)
    out    <- rep(NA_character_, n_max)
    k      <- min(n_max, length(counts))
    out[seq_len(k)] <- names(counts)[seq_len(k)]
    out
}

# Kept for compatibility: the nth most common song type (NA if the neighbour sang fewer than n different types)
get_nth_mode <- function(x, y, z, n = 2) get_top_modes(x, y, z, n_max = n)[n]

In [23]:
# The j-th most common song type of EACH NEIGHBOUR in the last <WINDOW_SIZE> seconds (j = 1..3)
# Fixes versus the earlier version:
#   1. the ranking j is used (it used the male number i for every j, so a male's three columns were identical)
#   2. the neighbours come from the focal male's own neighbourhood (a stale focal_neighbourhood was used before);
#      DgTbO is in two neighbourhoods, so each is handled separately
#   3. singer IDs are windowed together with the song types (slide2_index; the full-length ID column was passed
#      to every window before, so the neighbour's songs were not picked out correctly)
#   4. song types that were never sung give NA instead of an arbitrary unused factor level
WINDOW_SIZE <- 60
N_HIGHEST   <- 3

for (focal_id in focal_ids) {
    if (focal_id == "OPR") {next}
    print(focal_id)
    focal_nbs <- focal_males %>% filter(id == focal_id) %>% pull(neighbourhood)
    main_df   <- data_list[[focal_id]] %>% ungroup() %>% arrange(DateTime)
    new_cols  <- list()

    for (nb in focal_nbs) {
        in_nb <- !is.na(main_df$neighbourhood) & main_df$neighbourhood == nb
        for (i in 1:6) {
            neighbor_id <- focal_males %>% filter(neighbourhood == nb, male_number == as.character(i)) %>% pull(id)
            if ((neighbor_id == focal_id) || (neighbor_id == "OPR")) {next}

            # One pass per neighbour: his N_HIGHEST most common song types in each window
            tops <- slide_index2(
                main_df$UST, main_df$ID, main_df$DateTime,
                function(x, y) get_top_modes(x, y, neighbor_id, N_HIGHEST),
                .before = WINDOW_SIZE
            )
            for (j in 1:N_HIGHEST) {
                col <- sprintf("M%i_%i_highest_UST_%i_seconds", i, j, WINDOW_SIZE)
                if (is.null(new_cols[[col]])) new_cols[[col]] <- rep(NA_character_, nrow(main_df))
                new_cols[[col]][in_nb] <- map_chr(tops, j)[in_nb]
            }
        }
    }
    data_list[[focal_id]] <- bind_cols(main_df, as_tibble(new_cols))
}

[1] "DgTbO"
[1] "KLgK"
[1] "KRDg"
[1] "LbOW"
[1] "RRO"
[1] "RbDgY"
[1] "TbTbLg"
[1] "TbWY"
[1] "WLpTb"
[1] "YKY"


### Distance-weighted neighbour features

Adds distance-weighted versions of the neighbour features. Run after the window / last-song cells and before "Select relevant predictors".

**Weight** for a neighbour at distance `d` from the focal male (piecewise linear, never zero). `d` is `D_M<focal>_M<neighbour>` in the row of the neighbour's song (the distance when that neighbour sang), so it changes from song to song:
`w(d) = W_MIN + (1 - W_MIN) * max(0, 1 - d / D_MAX_M)`.
It is 1 at `d = 0`, falls linearly to `W_MIN` at `D_MAX_M`, and stays at `W_MIN` beyond that, so the farthest neighbour still contributes. The shape follows the song-type-sharing vs. distance plot in Logue et al. (2026).

**New columns** (all numeric):
- `n_songs_last60s_wtd`: weighted number of neighbour songs in the window
- `n_singers_last60s_wtd`: weighted number of neighbours that sang at least once in the window
- `n_ust_<type>_last60s_wtd`: weighted number of neighbour songs of each song type in the window
- `nbr_last_<k>_ust_<type>_wtd`: summed weights of the neighbours whose k-th most recent song is that type (each weighted by the distance when he sang that song)
- `n_songs_last60s`, `n_singers_last60s`: unweighted counterparts of the first two (optional)

Distances are missing when a bird has not been found yet that day. `MISSING_DISTANCE_RULE` sets how those neighbour songs are weighted (`pair_median` by default); the notebook prints how many songs were affected.


In [24]:
# ---- Distance-weighting parameters ------------------------------------------
# Shape read off the Logue et al. (2026) figure by eye: song-type sharing falls roughly linearly from
# ~0.55 at 50 m to ~0.10 at ~250 m, then is flat. Extrapolated to d = 0 the line is ~0.66, so 0.10 / 0.66 ~ 0.15.
D_MAX_M   <- 250   # distance at which the decline stops (same units as the D_M*_M* columns; assumed metres)
W_MIN     <- 0.15  # weight at and beyond D_MAX_M (keeps the farthest neighbours in)
LAST_LAGS <- 1:2   # neighbours' 1st and 2nd most recent song types (N_MAX = 3 are available)
ADD_UNWEIGHTED_COUNTERPARTS <- TRUE  # also add unweighted n_songs / n_singers (these two do not exist yet)
# A distance is missing when a bird has not been found yet that day. How to weight such neighbour songs:
#   "pair_median": use the pair's median distance over the rows where it is known (typical territory distance)
#   "same_day"   : nearest known value of the same pair on the same date; the pair median if none that day
#   "far"        : treat the neighbour as far away (weight = W_MIN)
#   "ignore"     : weight 0 (the song is left out of the weighted features)
#   "error"      : stop
MISSING_DISTANCE_RULE <- "pair_median"

distance_weight <- function(d, d_max = D_MAX_M, w_min = W_MIN) {
    w_min + (1 - w_min) * pmax(0, 1 - d / d_max)
}

# ---- Which D_M*_M* column holds each focal-neighbour distance -----------------
# Each neighbour song is weighted by the distance recorded in that song's own row
# (the birds move during the day, so the distance changes from song to song).
if (length(grep("^D_M[1-6]_M[1-6]$", names(raw_data))) == 0) stop("No D_M*_M* columns found in raw_data")

# Neighbours of one focal male in one neighbourhood, with the column holding their distance.
# Uses D_M<focal>_M<neighbour>, or D_M<neighbour>_M<focal> if only that direction exists.
neighbour_columns <- function(focal_id, nb) {
    members   <- focal_males %>%
        filter(neighbourhood == nb, id != "OPR") %>%   # OPR was removed from the data
        mutate(num = as.integer(male_number))
    focal_num <- members$num[members$id == focal_id]
    out <- members %>%
        filter(id != focal_id) %>%
        transmute(neighbour_id  = id,
                  neighbour_num = num,
                  dist_col = if_else(sprintf("D_M%d_M%d", focal_num, num) %in% names(raw_data),
                                     sprintf("D_M%d_M%d", focal_num, num),
                                     sprintf("D_M%d_M%d", num, focal_num)))
    missing <- !(out$dist_col %in% names(raw_data))
    if (any(missing)) {
        stop(sprintf("No distance column for focal male %s (%s) to: %s", focal_id, nb,
                     paste(out$neighbour_id[missing], collapse = ", ")))
    }
    out
}

# Inspect the distances (and the weights they give) in each neighbour's own song rows
focal_pairs  <- focal_males %>% filter(id %in% focal_ids)
weight_table <- map2_dfr(focal_pairs$neighbourhood, focal_pairs$id, function(nb, fid) {
    nbrs <- neighbour_columns(fid, nb)
    pmap_dfr(nbrs, function(neighbour_id, neighbour_num, dist_col) {
        rows <- raw_data %>% filter(neighbourhood == nb, ID == neighbour_id)   # rows where this neighbour sang
        d    <- rows[[dist_col]]
        tibble(focal_id = fid, neighbourhood = nb, neighbour_id = neighbour_id, dist_col = dist_col,
               n_songs = length(d), n_missing = sum(is.na(d)), pct_missing = round(100 * mean(is.na(d)), 1),
               d_min = min(d, na.rm = TRUE), d_median = median(d, na.rm = TRUE), d_max = max(d, na.rm = TRUE),
               w_at_d_min = distance_weight(d_min), w_at_d_max = distance_weight(d_max))
    })
})
weight_table

focal_id,neighbourhood,neighbour_id,dist_col,n_songs,n_missing,pct_missing,d_min,d_median,d_max,w_at_d_min,w_at_d_max
<chr>,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
LbOW,north,RbDgY,D_M1_M2,656,25,3.8,78.37111,94.03680,97.32587,0.7335382,0.6690920
LbOW,north,YKY,D_M1_M3,1064,127,11.9,31.94814,47.44059,61.87785,0.8913763,0.7896153
LbOW,north,TbTbLg,D_M1_M4,734,66,9.0,61.21936,67.67739,76.55291,0.7918542,0.7397201
LbOW,north,DgTbO,D_M1_M5,1167,144,12.3,52.95309,62.95599,89.35258,0.8199595,0.6962012
LbOW,north,RRO,D_M1_M6,535,19,3.6,41.65826,53.63760,104.82346,0.8583619,0.6436002
RbDgY,north,LbOW,D_M1_M2,699,247,35.3,78.37111,94.03680,94.03680,0.7335382,0.6802749
RbDgY,north,YKY,D_M2_M3,1064,365,34.3,33.24599,47.94988,84.78102,0.8869636,0.7117445
RbDgY,north,TbTbLg,D_M2_M4,734,323,44.0,42.09414,65.91978,86.89284,0.8568799,0.7045643
RbDgY,north,DgTbO,D_M2_M5,1167,435,37.3,138.82831,156.75710,181.83559,0.5279838,0.3817590


In [25]:
# ---- Distance-weighted neighbour features -----------------------------------
# Row index of one neighbour's k-th most recent song at every row (same date only; NA until he has sung k songs)
nth_last_index <- function(is_j, date, k) {
    tibble(row_id = seq_along(is_j), is_j = is_j, date = date) %>%
        group_by(date) %>%
        mutate(n_so_far = cumsum(is_j),
               idx = if_else(n_so_far >= k, row_id[is_j][pmax(n_so_far - k + 1L, 1L)], NA_integer_)) %>%
        pull(idx)
}

# Fill missing values of one distance column using the nearest earlier/later value on the same date
# (the column is a property of the pair, so any row of the neighbourhood can supply it)
fill_distance <- function(d, date, in_nb) {
    idx    <- which(in_nb)
    filled <- tibble(d = d[idx], date = date[idx]) %>%
        group_by(date) %>%
        tidyr::fill(d, .direction = "downup") %>%
        ungroup() %>%
        pull(d)
    d[idx] <- filled
    d
}

# df: all songs heard by the focal male in one neighbourhood setting, sorted by DateTime
# types: the focal male's repertoire (song types) as character
# Every neighbour song gets the weight of the distance recorded in THAT song's row (D_M<focal>_M<neighbour>).
# A neighbour's last-song type is weighted by the distance at the time he sang that song.
build_weighted_features <- function(df, focal_id, nb, types) {
    ust    <- as.character(df$UST)
    id_vec <- as.character(df$ID)
    in_nb  <- !is.na(df$neighbourhood) & df$neighbourhood == nb
    nbrs   <- neighbour_columns(focal_id, nb)
    n_nbr  <- nrow(nbrs)
    is_j   <- lapply(nbrs$neighbour_id, function(id) in_nb & id_vec == id)   # rows sung by each neighbour

    # Row-by-neighbour matrix of weights; only the rows a neighbour sang are ever used for him.
    # A neighbour's distance is missing on days his location is unknown; MISSING_DISTANCE_RULE says what to do then.
    W <- matrix(NA_real_, nrow = nrow(df), ncol = n_nbr)
    for (j in seq_len(n_nbr)) {
        d      <- df[[nbrs$dist_col[j]]]
        miss   <- is.na(d)
        n_na   <- sum(miss[is_j[[j]]])                   # missing distances on his own song rows
        pair_median <- median(d[in_nb], na.rm = TRUE)    # typical distance of this pair in this neighbourhood
        n_same_day  <- 0L

        if (n_na > 0) {
            if (MISSING_DISTANCE_RULE == "error") {
                stop(sprintf("Missing %s in %s's songs (%s): %d songs. Choose another MISSING_DISTANCE_RULE.",
                             nbrs$dist_col[j], nbrs$neighbour_id[j], nb, n_na))
            }
            if (MISSING_DISTANCE_RULE == "same_day") {
                d <- fill_distance(d, df$date, in_nb)
                n_same_day <- n_na - sum(is.na(d[is_j[[j]]]))
            }
            if (MISSING_DISTANCE_RULE %in% c("pair_median", "same_day")) {
                d[in_nb & is.na(d)] <- pair_median
            }
            distance_fill_log[[length(distance_fill_log) + 1]] <<- tibble(
                focal_id = focal_id, neighbourhood = nb, neighbour_id = nbrs$neighbour_id[j],
                dist_col = nbrs$dist_col[j], n_songs = sum(is_j[[j]]), n_missing = n_na,
                pct_missing = round(100 * n_na / sum(is_j[[j]]), 1), rule = MISSING_DISTANCE_RULE,
                n_filled_same_day = n_same_day, pair_median = pair_median)
        }

        w <- distance_weight(d)
        if (MISSING_DISTANCE_RULE == "far")    w[miss] <- W_MIN
        if (MISSING_DISTANCE_RULE == "ignore") w[miss] <- 0
        W[, j] <- w
        if (anyNA(W[is_j[[j]], j])) {
            stop(sprintf("No usable distance for %s in %s's songs (%s): the pair has no known distance at all.",
                         nbrs$dist_col[j], nbrs$neighbour_id[j], nb))
        }
    }
    w_j    <- lapply(seq_len(n_nbr), function(j) ifelse(is_j[[j]], W[, j], 0))  # weight on his own song rows, 0 elsewhere
    w_song <- Reduce(`+`, w_j)                                                   # weight of whichever neighbour sang the row

    in_window <- function(x) slide_index_sum(x, i = df$DateTime, before = WINDOW_SIZE)
    tag <- sprintf("last%is", WINDOW_SIZE)
    out <- list()

    # Songs and singers in the window
    n_j   <- lapply(is_j, function(ix) in_window(as.numeric(ix)))   # his number of songs in the window
    w_win <- lapply(w_j, in_window)                                 # summed weights of his songs in the window
    total_raw <- 0; singers_wtd <- 0; singers_raw <- 0
    for (j in seq_len(n_nbr)) {
        total_raw   <- total_raw   + n_j[[j]]
        singers_raw <- singers_raw + (n_j[[j]] > 0)
        singers_wtd <- singers_wtd + ifelse(n_j[[j]] > 0, w_win[[j]] / n_j[[j]], 0)  # his mean song weight in the window
    }
    out[[paste0("n_songs_", tag, "_wtd")]]   <- in_window(w_song)
    out[[paste0("n_singers_", tag, "_wtd")]] <- singers_wtd
    if (ADD_UNWEIGHTED_COUNTERPARTS) {
        out[[paste0("n_songs_", tag)]]   <- total_raw
        out[[paste0("n_singers_", tag)]] <- singers_raw
    }

    # Neighbours' song counts per song type in the window
    for (t in types) {
        out[[sprintf("n_ust_%s_%s_wtd", t, tag)]] <- in_window(w_song * (!is.na(ust) & ust == t))
    }

    # Neighbours' k-th most recent song type, weighted by the distance when he sang that song
    for (k in LAST_LAGS) {
        idx_j     <- lapply(is_j, function(ix) nth_last_index(ix, df$date, k))
        last_type <- lapply(idx_j, function(ix) ust[ix])
        last_w    <- lapply(seq_len(n_nbr), function(j) {
            w <- W[idx_j[[j]], j]
            w[is.na(w)] <- 0
            w
        })
        for (t in types) {
            total <- 0
            for (j in seq_len(n_nbr)) {
                total <- total + last_w[[j]] * (!is.na(last_type[[j]]) & last_type[[j]] == t)
            }
            out[[sprintf("nbr_last_%i_ust_%s_wtd", k, t)]] <- total
        }
    }
    as_tibble(out)
}

stopifnot(MISSING_DISTANCE_RULE %in% c("pair_median", "same_day", "far", "ignore", "error"))
distance_fill_log <- list()

for (focal_id in focal_ids) {
    print(focal_id)
    main_df <- data_list[[focal_id]] %>% ungroup() %>% arrange(DateTime)
    nbs     <- focal_males %>% filter(id == focal_id) %>% pull(neighbourhood)  # DgTbO is in two neighbourhoods

    # The focal male's own repertoire (others' out-of-repertoire songs were recoded to -1 earlier)
    types <- main_df %>% filter(ID == focal_id) %>% pull(UST) %>% as.character() %>% unique() %>% setdiff(c("0", "-1"))
    types <- types[order(as.numeric(types))]

    feats <- NULL
    for (nb in nbs) {
        f     <- build_weighted_features(main_df, focal_id, nb, types)
        in_nb <- !is.na(main_df$neighbourhood) & main_df$neighbourhood == nb
        if (is.null(feats)) feats <- f else feats[in_nb, ] <- f[in_nb, ]
    }
    data_list[[focal_id]] <- bind_cols(main_df, feats)
}

# Missing distances that had to be filled (empty if none)
if (length(distance_fill_log) == 0) {
    cat("No missing distances in any neighbour song row.\n")
} else {
    distance_fill_log <- bind_rows(distance_fill_log)
    cat(sprintf("Neighbour songs with a missing distance (rule = \"%s\"):\n", MISSING_DISTANCE_RULE))
    print(distance_fill_log, n = Inf, width = Inf)
}

# Quick look at the new columns for the first focal male
data_list[[1]] %>%
    filter(ID == focal_ids[1]) %>%
    select(DateTime, matches("^n_(songs|singers)_last[0-9]+s"), contains("_wtd")) %>%
    select(1:8) %>%
    head()

[1] "DgTbO"
[1] "KLgK"
[1] "KRDg"
[1] "LbOW"
[1] "RRO"
[1] "RbDgY"
[1] "TbTbLg"
[1] "TbWY"
[1] "WLpTb"
[1] "YKY"
Neighbour songs with a missing distance (rule = "pair_median"):
# A tibble: 41 × 10
   focal_id neighbourhood neighbour_id dist_col n_songs n_missing pct_missing
   <chr>    <chr>         <chr>        <chr>      <int>     <int>       <dbl>
 1 DgTbO    north         LbOW         D_M1_M5      699         2         0.3
 2 DgTbO    north         RbDgY        D_M2_M5      656        17         2.6
 3 DgTbO    north         YKY          D_M3_M5     1064        10         0.9
 4 DgTbO    north         TbTbLg       D_M4_M5      734         7         1  
 5 DgTbO    centre        TbWY         D_M1_M6      604         5         0.8
 6 DgTbO    centre        KRDg         D_M2_M6     1021         8         0.8
 7 KLgK     centre        TbWY         D_M1_M5      604       185        30.6
 8 KLgK     centre        KRDg         D_M2_M5     1021       289        28.3
 9 KLgK     centre     

DateTime,n_songs_last60s_wtd,n_singers_last60s_wtd,n_songs_last60s,n_singers_last60s,n_ust_1_last60s_wtd,n_ust_4_last60s_wtd,n_ust_5_last60s_wtd
<dttm>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2022-05-13 05:24:14,0.5938845,0.5938845,1,1,0,0,0.0000000
2022-05-13 05:24:40,1.1877690,0.5938845,2,1,0,0,0.0000000
2022-05-13 05:24:48,1.1877690,0.5938845,2,1,0,0,0.0000000
2022-05-13 05:24:59,1.8587348,1.2648503,3,2,0,0,0.6709659
2022-05-13 05:25:09,1.8587348,1.2648503,3,2,0,0,0.6709659
2022-05-13 05:25:18,2.5297007,1.2648503,4,2,0,0,0.6709659


In [26]:
# Select relevant predictors
for (focal_id in focal_ids) {
    if (focal_id == "OPR") {next}
    data_list[[focal_id]] <- data_list[[focal_id]] %>%
        filter(ID == focal_id) %>%
        select(
            ID,
            UST,
            DateTime, time_rel_sunrise,
            "D_M1_M2", "D_M1_M3", "D_M1_M4", "D_M1_M5", "D_M1_M6",
            contains("last_UST"), # Nth last song
            contains("last_song"), # Time since last song
            focal_male_song_number,
            time_since_M1_1_last_song,
            DateTime, time_rel_sunrise,
            contains("n_ust_"), # n_ust_X = Number of times song type X has been heard since M1's last song
            contains("matches_"),
            # time_M1_last_song, time_M2_last_song, time_M3_last_song, time_M4_last_song, time_M5_last_song, time_M6_last_song,
            # time_since_M1_last_song, time_since_M2_last_song, time_since_M3_last_song, time_since_M4_last_song, time_since_M5_last_song, time_since_M6_last_song, # did not include time since M1 last song because it is always 0
            number_of_matches, does_it_match,
            contains("shared_songs"),
            contains("Attention"),
            contains("attention"),
            contains("target"),
            contains("others"),
            contains("highest_UST"), # Nth Most common song types from each neighbour in time window
            contains("_wtd"), # Distance-weighted neighbour features
            matches("^n_(songs|singers)_last[0-9]+s$"), # Unweighted counterparts of the above
            male_number
        ) %>% filter(UST != 0)
}

In [27]:
# Expand each focal song into one row per candidate song type in the focal male's repertoire
# Original song-type columns and counts are retained; candidate-relative columns are added.

for (focal_id in focal_ids) {
    if (focal_id == "OPR") {next}

    focal_data <- data_list[[focal_id]] %>%
        ungroup() %>%
        filter(ID == focal_id) %>%
        mutate(event_id = paste(focal_id, row_number(), sep = "_"))
    focal_data <- focal_data %>%
        mutate(UST = as.character(UST))
    repertoire <- focal_data %>%
        pull(UST) %>%
        unique() %>%
        na.omit() %>%
        .[. != "0"] %>%
        sort()

    candidate_data <- map_dfr(repertoire, function(candidate_type) {
        candidate_rows <- focal_data %>%
            mutate(
                candidate_ust = candidate_type,
                match = as.integer(UST == candidate_type)
            )

        count_columns <- names(candidate_rows)[str_detect(names(candidate_rows), "^n_ust_[0-9]+$")]
        window_columns <- names(candidate_rows)[str_detect(names(candidate_rows), "^n_ust_[0-9]+_window$")]
        target_columns <- names(candidate_rows)[str_detect(names(candidate_rows), "^n_ust_[0-9]+_target_window$")]
        others_columns <- names(candidate_rows)[str_detect(names(candidate_rows), "^n_ust_[0-9]+_others_window$")]

        candidate_columns <- c(
            candidate_n_ust = sprintf("n_ust_%s", candidate_type),
            candidate_n_ust_window = sprintf("n_ust_%s_window", candidate_type),
            candidate_n_ust_target_window = sprintf("n_ust_%s_target_window", candidate_type),
            candidate_n_ust_others_window = sprintf("n_ust_%s_others_window", candidate_type)
        )
        for (candidate_column in names(candidate_columns)) {
            source_column <- candidate_columns[[candidate_column]]
            candidate_rows[[candidate_column]] <- if (source_column %in% names(candidate_rows)) {
                candidate_rows[[source_column]]
            } else {
                NA_real_
            }
        }

        song_type_columns <- names(candidate_rows)[str_detect(names(candidate_rows), "UST")]
        song_type_columns <- setdiff(song_type_columns, c("UST", "candidate_ust", count_columns, window_columns, target_columns, others_columns))
        for (column in song_type_columns) {
            candidate_rows[[paste0("candidate_matches_", column)]] <- as.integer(as.numeric(as.character(candidate_rows[[column]])) == candidate_type)
        }

        candidate_rows
    })

    data_list[[focal_id]] <- candidate_data
}

In [28]:
for (focal_id in focal_ids) {
    if (focal_id == "OPR") {next}
    event_checks <- data_list[[focal_id]] %>%
        group_by(event_id) %>%
        summarise(n_candidates = n(), n_matches = sum(match == 1), .groups = "drop")
    if (any(event_checks$n_matches != 1)) {
        stop(sprintf("Invalid candidate events for %s", focal_id))
    }
    write_csv(data_list[[focal_id]], sprintf("data/by_id/%s_candidate_classification.csv", focal_id))
}